In [1]:
import pandas as pd
from datasets import Dataset
import numpy as np
import random
import torch
import os
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments
from transformers import Trainer

In [2]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)

# Make constant variables.
MODEL_NAME = "distilbert-base-uncased"

In [3]:
# Load the dataset
df = pd.read_csv("./dataSyntheticAll.csv")
dataset = Dataset.from_pandas(df)

In [4]:
# Map dataset labels to 1s or 0s.
label_map = {
    "met": 0,
    "unmet": 1
}

dataset = dataset.map(lambda x: {"label": label_map[x["needs"]]})

Map:   0%|          | 0/5783 [00:00<?, ? examples/s]

In [5]:
# Tokenize texts.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(example):
    return tokenizer(
        example["report"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

dataset = dataset.map(tokenize)
 # Set PyTorch format. 
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

Map:   0%|          | 0/5783 [00:00<?, ? examples/s]

In [6]:
# Split the dataset.
dataset = dataset.train_test_split(test_size=0.1, seed=RANDOM_STATE)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [7]:
# Confirm device for use. 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
# Load the base model.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# Define LoRA config.
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_lin", "v_lin"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS"
)

# Attach LoRA to model. 
model = get_peft_model(model, lora_config)
model.to(device)
model.print_trainable_parameters()

Device: cuda


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 739,586 || all params: 67,694,596 || trainable%: 1.0925


In [8]:
# Set training arguments.
training_args = TrainingArguments(
    output_dir="synthetic_data_model",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True,
    fp16=True
)

# Create trainer.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [9]:
# Train the model.
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.288641,0.361126
2,0.291273,0.322208
3,0.280247,0.317794


TrainOutput(global_step=3903, training_loss=0.3098660175109441, metrics={'train_runtime': 2073.3981, 'train_samples_per_second': 7.53, 'train_steps_per_second': 1.882, 'total_flos': 1051775809855488.0, 'train_loss': 0.3098660175109441, 'epoch': 3.0})

In [10]:
# Save the fine-tuned model.
model.save_pretrained("synth_lora_model")
tokenizer.save_pretrained("synth_lora_model")

('synth_lora_model/tokenizer_config.json', 'synth_lora_model/tokenizer.json')